# Lab 7 — LeNet for PneumoniaMNIST
Streamlined Colab edition. Full commented version is in the course Google Drive folder. Educational use only.

In [ ]:
!pip -q install medmnist
import numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import medmnist
from medmnist import INFO
from sklearn.metrics import confusion_matrix,accuracy_score,recall_score,precision_score,f1_score,roc_auc_score
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print(device)

In [ ]:
flag='pneumoniamnist'; info=INFO[flag]; DataClass=getattr(medmnist,info['python_class'])
tf=transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.5,),(0.5,))])
train=DataClass(split='train',transform=tf,download=True); val=DataClass(split='val',transform=tf,download=True); test=DataClass(split='test',transform=tf,download=True)
train_loader=DataLoader(train,batch_size=64,shuffle=True); val_loader=DataLoader(val,batch_size=128); test_loader=DataLoader(test,batch_size=128)
print(len(train),len(val),len(test))

In [ ]:
class LeNet(nn.Module):
 def __init__(self):
  super().__init__(); self.features=nn.Sequential(nn.Conv2d(1,6,5,padding=2),nn.ReLU(),nn.AvgPool2d(2),nn.Conv2d(6,16,5),nn.ReLU(),nn.AvgPool2d(2)); self.classifier=nn.Sequential(nn.Flatten(),nn.Linear(16*5*5,120),nn.ReLU(),nn.Linear(120,84),nn.ReLU(),nn.Linear(84,2))
 def forward(self,x): return self.classifier(self.features(x))
model=LeNet().to(device); loss_fn=nn.CrossEntropyLoss(); opt=optim.Adam(model.parameters(),lr=1e-3)

In [ ]:
for epoch in range(3):
 model.train(); total=correct=0; running=0
 for x,y in train_loader:
  x=x.to(device); y=y.squeeze().long().to(device); opt.zero_grad(); z=model(x); loss=loss_fn(z,y); loss.backward(); opt.step(); running+=loss.item()*len(y); correct+=(z.argmax(1)==y).sum().item(); total+=len(y)
 print(f'Epoch {epoch+1}: loss={running/total:.4f}, acc={correct/total:.3f}')

In [ ]:
model.eval(); yt=[]; yp=[]; ps=[]
with torch.no_grad():
 for x,y in test_loader:
  z=model(x.to(device)); p=torch.softmax(z,1)[:,1].cpu().numpy(); pred=z.argmax(1).cpu().numpy(); yt.extend(y.squeeze().numpy()); yp.extend(pred); ps.extend(p)
cm=confusion_matrix(yt,yp); tn,fp,fn,tp=cm.ravel(); print(cm); print('Accuracy',accuracy_score(yt,yp)); print('Sensitivity',recall_score(yt,yp)); print('Specificity',tn/(tn+fp)); print('F1',f1_score(yt,yp)); print('ROC-AUC',roc_auc_score(yt,ps))